## Демо 3 - Debezium + JDBC sink

Переливаем данные из одной БД в другую через Kafka:

`postgres_master` -> Debezium (source) -> Kafka -> Debezium (JDBC sink) -> `postgres_dwh`

```bash
docker-compose up -d
```

* master: `localhost:5432`, в нем 9 таблиц магазина (см. `createdb.sql` и `filldb.sql`)
* dwh: `localhost:5434`, пока пустой
* Debezium UI: http://localhost:8080

In [ ]:
from time import sleep, time

import psycopg2
import requests

CONNECT_URL = "http://localhost:8083/connectors"

master = psycopg2.connect(host="localhost", port=5432, user="postgres", password="postgres", dbname="postgres")
dwh = psycopg2.connect(host="localhost", port=5434, user="postgres", password="postgres", dbname="postgres")
master.autocommit = True
dwh.autocommit = True


def sql(conn, query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            return cur.fetchall()


def print_table(header, rows):
    widths = [max(len(str(x)) for x in column) for column in zip(header, *rows)]
    for row in [header] + rows:
        print("  ".join(str(x).ljust(w) for x, w in zip(row, widths)))


def wait_for(check, timeout=60):
    # Повторяем check раз в секунду, пока он не вернет что-то непустое
    deadline = time() + timeout
    while time() < deadline:
        result = check()
        if result:
            return result
        sleep(1)
    raise TimeoutError("Не дождались")

### Шаг 1. Описываем коннекторы

Один source: читает WAL из master и пишет изменения в топики `postgres.public.<таблица>`.<br>
И по одному sink на таблицу: читает свой топик и делает upsert в DWH.

In [ ]:
pg_connector = {
    "name": "pg-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "plugin.name": "pgoutput",

        "database.hostname": "postgres_master",
        "database.port": "5432",
        "database.user": "postgres",
        "database.password": "postgres",
        "database.dbname" : "postgres",

        "table.include.list": "public.(.*)",
        "heartbeat.interval.ms": "5000",
        "slot.name": "dbname_debezium",
        "publication.name": "dbname_publication",

        "topic.creation.default.cleanup.policy": "delete",
        "topic.creation.default.partitions": "1",
        "topic.creation.default.replication.factor": "1",
        "topic.creation.default.retention.ms": "604800000",
        "topic.creation.enable": "true",
        "topic.prefix": "postgres"
    }
}

tables = [
    'categories', 'customers', 'manufacturers', 'deliveries', 'price_change',
    'products', 'purchase_items', 'purchases', 'stores',
]


def sink_connector(table):
    topic = f"postgres.public.{table}"
    return {
        "name": topic.replace('.', '-'),
        "config": {
            "connector.class": "io.debezium.connector.jdbc.JdbcSinkConnector",
            "topics": topic,
            "connection.url": "jdbc:postgresql://postgres_dwh:5432/postgres",
            "connection.username": "postgres",
            "connection.password": "postgres",
            "tasks.max": "1",
            "insert.mode": "upsert",
            # Удаления в DWH не переносим, подробнее в шаге 4
            "delete.enabled": "false",
            # Первичный ключ берем из ключа сообщения, то есть из PK таблицы в master
            "primary.key.mode": "record_key",
            # Таблицы в DWH создаются сами
            "schema.evolution": "basic"
        }
    }

### Шаг 2. Регистрируем

Через REST API Kafka Connect. PUT создает коннектор, а если он уже есть - обновляет,
поэтому ячейки можно запускать повторно.

In [ ]:
def put_connector(connector):
    response = requests.put(f"{CONNECT_URL}/{connector['name']}/config", json=connector["config"])
    assert response.status_code < 400, response.text

In [ ]:
put_connector(pg_connector)

In [ ]:
for table in tables:
    put_connector(sink_connector(table))

Смотрим, что все запустилось (то же самое есть в Debezium UI)

In [ ]:
def connectors_status():
    rows = []
    for name in sorted(requests.get(CONNECT_URL).json()):
        status = requests.get(f"{CONNECT_URL}/{name}/status").json()
        tasks = [task["state"] for task in status["tasks"]]
        rows.append((name, status["connector"]["state"], ", ".join(tasks) or "-"))
    return rows


def running_connectors():
    # Отдаем статусы, только когда все коннекторы и их задачи в RUNNING
    rows = connectors_status()
    if all(state == "RUNNING" and tasks == "RUNNING" for _, state, tasks in rows):
        return rows


rows = wait_for(running_connectors)
print_table(["connector", "state", "tasks"], rows)

### Шаг 3. Данные доехали

Сравниваем число строк в master и в DWH.<br>
Таблицы в DWH называются как топики: `postgres_public_<таблица>`, sink создал их сам.

In [ ]:
def count(conn, table):
    try:
        return sql(conn, f"select count(*) from public.{table}")[0][0]
    except psycopg2.errors.UndefinedTable:
        return None  # sink еще не создал таблицу


def counts():
    return [(t, count(master, t), count(dwh, f"postgres_public_{t}")) for t in tables]


def caught_up_counts():
    # Отдаем счетчики, когда в DWH строк уже не меньше, чем в master
    # (не "поровну": удаления мы не переносим, при повторном запуске DWH может быть больше)
    rows = counts()
    if all(d is not None and d >= m for _, m, d in rows):
        return rows


rows = wait_for(caught_up_counts)
print_table(["table", "master", "dwh"], rows)

### Шаг 4. Живые изменения

Меняем данные в master и смотрим, что происходит в DWH

In [ ]:
def show_customer(customer_id):
    query = "select customer_id, customer_fname, customer_phone from public.{} where customer_id = %s"
    print("master:", sql(master, query.format("customers"), (customer_id,)))
    print("dwh:   ", sql(dwh, query.format("postgres_public_customers"), (customer_id,)))

**Insert**

In [ ]:
new_id = sql(master, """
    insert into public.customers (customer_fname, customer_lname, customer_gender, customer_phone)
    values ('Ivan', 'Petrov', 'man', '89990001122')
    returning customer_id
""")[0][0]

wait_for(lambda: sql(dwh, "select 1 from public.postgres_public_customers where customer_id = %s", (new_id,)))
show_customer(new_id)

**Update**

In [ ]:
sql(master, "update public.customers set customer_phone = '80000000000' where customer_id = %s", (new_id,))

wait_for(lambda: sql(dwh, "select 1 from public.postgres_public_customers where customer_id = %s and customer_phone = '80000000000'", (new_id,)))
show_customer(new_id)

**Delete**

In [ ]:
sql(master, "delete from public.customers where customer_id = %s", (new_id,))

sleep(5)  # insert и update доезжали за секунду, с запасом хватит
show_customer(new_id)

В master строки нет, а в DWH она осталась.<br>
Так мы настроили sink: `delete.enabled: false`, delete-события он пропускает.
Для DWH это часто нормально: история не теряется.

In [ ]:
rows = counts()
print_table(["table", "master", "dwh"], rows)

В `customers` теперь разница в одну строку.<br>
Если нужно, чтобы удаления доезжали, в sink ставим `delete.enabled: true`
(тогда в DWH строка тоже удалится).